# 02 - Behavioural analysis

Stage 2: build past-only behavioural features and check they are defined often enough to be useful.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

from src import config, preprocessing, data_profiling, feature_engineering
from src.utils import setup_logging
setup_logging()

# Use --nrows style subsetting here to keep the notebook responsive.
# PaySim is time-ordered, so this is an EARLY TIME WINDOW, not a random sample.
df = preprocessing.load_paysim(nrows=1_000_000)
df = preprocessing.prepare(df)
df.head()

In [ ]:
featured, feature_columns = feature_engineering.build_features(df)
print(f'{len(feature_columns)} features built')
feature_columns

## Coverage: is each feature actually defined?

A feature that is NaN for 97% of rows is not a feature, it is a footnote. This table is what stops fabricated customer-history signals reaching the model.


In [ ]:
coverage = feature_engineering.feature_coverage(featured, feature_columns)
coverage.head(20)

## Do the features separate the classes?

If the fraud and legitimate medians are nearly identical, the feature will carry little weight. Cheap check before spending an hour on training.


In [ ]:
compare = (featured.groupby(config.TARGET)[[
    'amount', 'dest_prior_txn_count', 'dest_prior_unique_senders',
    'dest_txn_prev_24h', 'dest_txn_prev_168h', 'pair_prior_txn_count',
    'pair_is_new', 'dest_is_new', 'dest_is_merchant']]
    .median().T)
compare.columns = ['legitimate', 'fraudulent']
compare

## Leakage self-check

For the first transaction of any account, every prior-history column must be zero. If not, a window frame is including the current row.


In [ ]:
first_seen = featured.groupby('dest_id', sort=False).cumcount() == 0
violations = int((featured.loc[first_seen, 'dest_prior_txn_count'] != 0).sum())
print(f'frames that include the current row: {violations}  (expect 0)')